## 03 — Analyze & Salvage Failed BIDS Conversions

In `02-organize-in-bids-format.ipynb`, **3,242 / 3,278** fMRI–T1w session pairs converted cleanly into BIDS format (`data/raw_bids/`), leaving **36 failed conversions**:
1. **1 failed T1w scan (`130_S_2373`, `ses-20110502`, paired with fMRI `I232243`):**
   - Image ID `I232236` was misassigned in the master table; it is actually a `3_Plane_Localizer` with multi-orientation 2D scout slices (`dcm2niix` return code `1`).
   - The true 3D anatomical `MPRAGE` scan for the same subject and session is **`I232233`** (`data/raw_smri/ADNI/130_S_2373/MPRAGE/2011-05-02_08_52_04.0/I232233`).
2. **35 failed fMRI scans (`9` GE Medical Systems + `26` Philips Medical Systems):**
   - Failed `dcm2niix` with return code `1` due to either an erroneous per-slice GE private tag `(0019,10A2)` (`Raw data run number`), isolated single-slice PACS transfer dropouts, or truncated series uploads.
   
This notebook:
1. Updates `clean_dedup_fmri_to_t1_master.csv` to use **`I232233`** (`MPRAGE`) for fMRI record **`I232243`**, converts `I232233` into BIDS (`sub-130S2373_ses-20110502_T1w.nii.gz` + `.json`), and updates `bids_conversion_manifest.csv` (**bringing T1w conversion to 3,278 / 3,278 = 100.00%**).
2. Performs a comprehensive DICOM header analysis across all **35 failed fMRI series** and classifies each scan by its exact `(Slice × Timepoint)` completeness matrix.
3. Executes a targeted, non-destructive salvage pipeline (staging into a temporary directory on `D:\` without altering raw DICOMs) that recovers **24 / 35 failed fMRI scans** into valid 4D BIDS NIfTI + JSON files, and documents the **11 truly unrecoverable scans** where severe slice-major upload truncation removed `15%–98%` of the brain anatomy.

In [ ]:
import gzip
import json
import os
import shutil
import struct
import subprocess
import time
import warnings
import _winapi
from collections import Counter, defaultdict
from pathlib import Path

import dcm2niix
import numpy as np
import pandas as pd
import pydicom
from pydicom.uid import generate_uid

warnings.filterwarnings("ignore", category=UserWarning, module="pydicom")

PROJECT_ROOT = Path(r"d:\Mahdi\OU\Alzheimer's Disease\gnn-connectivity-matrix")
RAW_FMRI_DIR = PROJECT_ROOT / "data" / "raw_fmri" / "ADNI"
RAW_SMRI_DIR = PROJECT_ROOT / "data" / "raw_smri" / "ADNI"
RAW_BIDS_DIR = PROJECT_ROOT / "data" / "raw_bids"
MASTER_CSV = (
    PROJECT_ROOT / "results" / "dataset_creation" / "clean_dedup_fmri_to_t1_master.csv"
)
MANIFEST_CSV = (
    PROJECT_ROOT / "results" / "dataset_creation" / "bids_conversion_manifest.csv"
)
DCM2NIIX_BIN = dcm2niix.bin


def read_nifti_dims(nii_gz_path: Path):
    with gzip.open(nii_gz_path, "rb") as f:
        hdr = f.read(348)
    dims = struct.unpack("<8h", hdr[40:56])
    return dims[1 : dims[0] + 1]


master_df = pd.read_csv(MASTER_CSV)
manifest_df = pd.read_csv(MANIFEST_CSV)
both_ok = manifest_df["fMRI_Converted"] & manifest_df["T1_Converted"]

print(f"Loaded master table:   {len(master_df):,} rows")
print(f"Loaded BIDS manifest:  {len(manifest_df):,} rows")
print(
    f"Initial fMRI converted: {manifest_df['fMRI_Converted'].sum():,} / {len(manifest_df):,} ({(manifest_df['fMRI_Converted'].mean() * 100):.2f}%)"
)
print(
    f"Initial T1w converted:  {manifest_df['T1_Converted'].sum():,} / {len(manifest_df):,} ({(manifest_df['T1_Converted'].mean() * 100):.2f}%)"
)
print(
    f"Initial complete pairs: {both_ok.sum():,} / {len(manifest_df):,} ({(both_ok.mean() * 100):.2f}%)"
)

Loaded master table:   3,278 rows
Loaded BIDS manifest:  3,278 rows
Initial fMRI converted: 3,243 / 3,278 (98.93%)
Initial T1w converted:  3,277 / 3,278 (99.97%)
Initial complete pairs: 3,242 / 3,278 (98.90%)


## 1. Fix Misassigned T1w Image (`I232236` $\rightarrow$ `I232233`) for fMRI Record `I232243`

For subject `130_S_2373` (`ses-20110502`, fMRI `I232243`), we update `clean_dedup_fmri_to_t1_master.csv` to point to the true 3D `MPRAGE` scan **`I232233`**, run `dcm2niix` to generate `sub-130S2373_ses-20110502_T1w.nii.gz` and `.json`, and update `bids_conversion_manifest.csv`.

In [ ]:
# 1. Update clean_dedup_fmri_to_t1_master.csv for Image Data ID == 'I232243'
mask_master = master_df["Image Data ID"] == "I232243"
assert mask_master.sum() == 1, f"Expected 1 row for I232243, found {mask_master.sum()}"

old_t1_id = master_df.loc[mask_master, "T1_Image_ID"].values[0]
old_t1_desc = master_df.loc[mask_master, "T1_Description"].values[0]
print(
    f"Before update -> fMRI: I232243 | T1_Image_ID: {old_t1_id} | T1_Description: {old_t1_desc}"
)

master_df.loc[mask_master, "T1_Image_ID"] = "I232233"
master_df.loc[mask_master, "T1_Raw_ID"] = 232233
master_df.loc[mask_master, "T1_Description"] = "MPRAGE"
master_df.to_csv(MASTER_CSV, index=False)
print(
    "Updated clean_dedup_fmri_to_t1_master.csv with T1_Image_ID = 'I232233' (MPRAGE)."
)

# 2. Convert I232233 to BIDS anat directory
t1_src_dir = next((RAW_SMRI_DIR / "130_S_2373").rglob("I232233"))
anat_dir = RAW_BIDS_DIR / "sub-130S2373" / "ses-20110502" / "anat"
anat_dir.mkdir(parents=True, exist_ok=True)
t1_prefix = "sub-130S2373_ses-20110502_T1w"

for old_f in anat_dir.glob(f"{t1_prefix}*"):
    old_f.unlink()

res_t1 = subprocess.run(
    [
        DCM2NIIX_BIN,
        "-z",
        "y",
        "-b",
        "y",
        "-f",
        t1_prefix,
        "-o",
        str(anat_dir),
        str(t1_src_dir),
    ],
    capture_output=True,
    text=True,
)
t1_nii = anat_dir / f"{t1_prefix}.nii.gz"
t1_json = anat_dir / f"{t1_prefix}.json"
assert res_t1.returncode == 0 and t1_nii.exists() and t1_json.exists(), (
    f"T1w conversion failed: {res_t1.stdout} {res_t1.stderr}"
)

t1_dims = read_nifti_dims(t1_nii)
print(f"Converted I232233 -> {t1_nii.relative_to(PROJECT_ROOT)} | dims = {t1_dims}")

# 3. Update bids_conversion_manifest.csv
mask_man = manifest_df["fMRI_Image_ID"] == "I232243"
manifest_df.loc[mask_man, "T1_Image_ID"] = "I232233"
manifest_df.loc[mask_man, "T1_Converted"] = True
manifest_df.loc[mask_man, "T1_Status"] = (
    "OK (Replaced I232236 localizer with I232233 MPRAGE)"
)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print(
    f"Updated T1w converted count: {manifest_df['T1_Converted'].sum():,} / {len(manifest_df):,} (100.00%)"
)

Before update -> fMRI: I232243 | T1_Image_ID: I232233 | T1_Description: MPRAGE
Updated clean_dedup_fmri_to_t1_master.csv with T1_Image_ID = 'I232233' (MPRAGE).
Converted I232233 -> data\raw_bids\sub-130S2373\ses-20110502\anat\sub-130S2373_ses-20110502_T1w.nii.gz | dims = (170, 256, 256)
Updated T1w converted count: 3,278 / 3,278 (100.00%)


## 2. Deep DICOM Header Diagnostic of the 35 Failed fMRI Series

We inspect the DICOM headers of all `.dcm` files across the 35 failed fMRI series (`stop_before_pixels=True`), mapping every file onto its `(Axial Slice Z, TemporalPositionIdentifier)` grid and checking for GE private tag `(0019,10A2)` (`Raw data run number`).

This reveals **5 root-cause categories**:
1. **`Cat 1 — GE Private Tag (0019,10A2) Bug` (5 scans, Salvageable $\rightarrow$ 200 vols):**
   - Site `007` GE Signa HDxt scans (`I680900`, `I638776`, `I631785`, `I695057`, `I680924`) have **all 9,600 files (`48 slices × 200 volumes = 100.0%`)**, but GE private tag `(0019,10A2)` was incremented per slice (`48` unique values instead of `1`), confusing `dcm2niix`'s volume indexer.
2. **`Cat 2 — Full-Brain (48 Slices) with 1–9 Isolated Single-Slice Dropouts` (13 scans: 11 Philips + 2 GE, Salvageable $\rightarrow$ 197–200 vols):**
   - `99.91%–99.99%` of all slices (`9,448–9,599` out of `9,456/9,600` files) are present; only `1` to `9` isolated single 2D slice files (`0.01%–0.09%`) were dropped during PACS transfer (plus 1 extra `TPI=None` scout slice in `I1158788`). Temporal linear interpolation of the `1–9` missing single slices from immediate neighbors `(t-1, t+1)` (plus removing `(0019,10A2)` on the 2 GE scans) recovers the complete `197` or `200` volume series.
3. **`Cat 3 — Full-Brain (48 Slices) Truncated at End or Burst Dropouts` (4 scans: 2 GE + 2 Philips, Salvageable $\rightarrow$ 98–167 vols):**
   - `009_S_4388 I880134` (GE, truncated during volume 168 $\rightarrow$ `167` complete vols), `027_S_6733 I1386024` (GE, truncated after volume 137 $\rightarrow$ `137` complete vols), `006_S_6689 I1563081` (Philips, multi-slice dropouts in early volumes $\rightarrow$ `166` contiguous complete vols `TPI 32..197`), and `019_S_6533 I1037522` (Philips, 34 single-slice dropouts at ends $\rightarrow$ `98` contiguous complete vols `TPI 27..124`). Staging only the contiguous complete volumes `[Best_Start..Best_End]` recovers clean 48-slice 4D volumes with zero interpolation.
4. **`Cat 4 — Near-Complete Axial Slab (44–47 Contiguous Slices, Full Cortex)` (2 Philips scans, Salvageable $\rightarrow$ 132–200 vols):**
   - Philips exports 2D-slice fMRI in **slice-major order** (all timepoints of slice 1, then slice 2, etc.). In `019_S_6315 I1314046` (`97.23%` files, `47/48` slices covering `Z = -67.8..+88.2 mm` across `TPI 1..132`) and `002_S_0413 I569636` (`92.26%` files, `44/48` slices covering `Z = -58.2..+84.2 mm` across all `200` volumes), only `1–4` extreme vertex/scalp slices (`Z > +84 mm`) were truncated, leaving a spatially contiguous cortical slab with `132` and `200` complete volumes.
5. **`Cat 5 — Unsalvageable (Severe Z-Axis Truncation or Internal Cortical Gap)` (11 Philips scans, Unusable):**
   - `10` Philips scans were truncated in slice-major order with only `1` to `41` axial slices present (`1.9%–85.4%` of files, missing `23 mm` to `155 mm` of brain tissue) or massive random dropouts (`I659915`, only 2 contiguous volumes), and `1` scan (`006_S_6500 I1028502`) is missing two internal slices inside the cortex (`Z = 70.40 mm` to `80.57 mm`, a `10.17 mm` internal spatial gap).

In [ ]:
def find_longest_contiguous_run(sorted_ints):
    if not sorted_ints:
        return (None, None, 0)
    best_start = cur_start = sorted_ints[0]
    best_end = cur_end = sorted_ints[0]
    for x in sorted_ints[1:]:
        if x == cur_end + 1:
            cur_end = x
        else:
            if (cur_end - cur_start + 1) > (best_end - best_start + 1):
                best_start, best_end = cur_start, cur_end
            cur_start = cur_end = x
    if (cur_end - cur_start + 1) > (best_end - best_start + 1):
        best_start, best_end = cur_start, cur_end
    return (best_start, best_end, best_end - best_start + 1)


from concurrent.futures import ThreadPoolExecutor

failed_fmri_df = manifest_df[~manifest_df["fMRI_Converted"]].copy()
print(f"Analyzing all {len(failed_fmri_df)} failed fMRI series...")


def analyze_single_series(row):
    subj = row["Subject"]
    ses = row["BIDS_Session"]
    img_id = row["fMRI_Image_ID"]
    dcm_dir = next((RAW_FMRI_DIR / subj).rglob(img_id))
    dcm_files = sorted(dcm_dir.glob("*.dcm"))

    z_to_tpis = defaultdict(set)
    tpi_to_zs = defaultdict(set)
    none_tpi_files = 0
    manufacturer = None
    ntp_vals = set()
    ge_10a2_vals = set()

    for f in dcm_files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        if manufacturer is None:
            manufacturer = str(getattr(ds, "Manufacturer", "")).strip()
        tpi = getattr(ds, "TemporalPositionIdentifier", None)
        ntp = getattr(ds, "NumberOfTemporalPositions", None)
        if ntp is not None:
            ntp_vals.add(int(ntp))
        if (0x0019, 0x10A2) in ds:
            ge_10a2_vals.add(ds[(0x0019, 0x10A2)].value)
        ipp = getattr(ds, "ImagePositionPatient", None)
        if tpi is None or ipp is None:
            none_tpi_files += 1
        else:
            z = round(float(ipp[2]), 2)
            z_to_tpis[z].add(int(tpi))
            tpi_to_zs[int(tpi)].add(z)

    declared_ntp = sorted(ntp_vals)[0] if ntp_vals else len(tpi_to_zs)
    valid_zs = sorted(z_to_tpis.keys())
    num_unique_z = len(valid_zs)
    z_diffs = set(np.round(np.diff(valid_zs), 1)) if num_unique_z > 1 else set()
    uniform_z = len(z_diffs) == 1

    # Standard slice count per volume (48 for full brain, or mode of slices per TPI)
    slices_per_tpi = [len(zs) for zs in tpi_to_zs.values()]
    mode_slices = Counter(slices_per_tpi).most_common(1)[0][0] if slices_per_tpi else 0
    complete_48_tpis = sorted([t for t, zs in tpi_to_zs.items() if len(zs) == 48])
    c48_start, c48_end, c48_len = find_longest_contiguous_run(complete_48_tpis)

    mode_complete_tpis = sorted(
        [t for t, zs in tpi_to_zs.items() if len(zs) == mode_slices]
    )
    cm_start, cm_end, cm_len = find_longest_contiguous_run(mode_complete_tpis)

    valid_dcm_count = len(dcm_files) - none_tpi_files
    expected_full_dcm = 48 * declared_ntp
    missing_from_full = expected_full_dcm - valid_dcm_count
    pct_complete = 100.0 * valid_dcm_count / expected_full_dcm

    # Classify into Categories 1..5
    if num_unique_z == 48 and missing_from_full == 0 and len(ge_10a2_vals) > 1:
        category = "Cat 1: GE (0019,10A2) Bug (100% Complete)"
        salvage_method = "strip_ge_10a2"
        salvageable = True
        target_dims = f"64x64x48x{declared_ntp}"
    elif num_unique_z == 48 and 1 <= missing_from_full <= 9:
        category = f"Cat 2: 48-Slice, {missing_from_full} Isolated Slice Dropout(s)"
        salvage_method = "interp_single_slices"
        salvageable = True
        target_dims = f"64x64x48x{declared_ntp}"
    elif num_unique_z == 48 and c48_len >= 90:
        category = f"Cat 3: 48-Slice, Contiguous Subset [{c48_start}..{c48_end}]"
        salvage_method = "contiguous_48_subset"
        salvageable = True
        target_dims = f"64x64x48x{c48_len}"
    elif (
        num_unique_z >= 44
        and uniform_z
        and valid_zs[-1] >= 84.0
        and pct_complete >= 90.0
    ):
        if img_id == "I1314046":
            category = "Cat 4: Near-Complete Slab (47/48 Slices, TPI 1..132)"
            salvage_method = "slab_tpi_subset"
            salvageable = True
            target_dims = "64x64x47x132"
        else:
            category = "Cat 4: Near-Complete Slab (44/48 Slices, All 200 Vols)"
            salvage_method = "slab_z_subset"
            salvageable = True
            target_dims = f"64x64x44x{declared_ntp}"
    else:
        if not uniform_z:
            reason = f"Internal Z-gap (dz={sorted(z_diffs)} mm)"
        elif num_unique_z < 44:
            reason = (
                f"Z-truncated ({num_unique_z}/48 slices, {pct_complete:.1f}% files)"
            )
        else:
            reason = (
                f"Severely incomplete ({valid_dcm_count}/{expected_full_dcm} = {pct_complete:.1f}% files)"
                if pct_complete < 50.0
                else f"Scattered dropouts (max {cm_len} contig vols)"
            )
        category = f"Cat 5: Unsalvageable — {reason}"
        salvage_method = "unsalvageable"
        salvageable = False
        target_dims = "N/A"

    return {
        "Subject": subj,
        "BIDS_Session": ses,
        "fMRI_Image_ID": img_id,
        "Vendor": "GE" if "GE" in manufacturer.upper() else "Philips",
        "Total_DCM": len(dcm_files),
        "Expected_DCM": expected_full_dcm,
        "Pct_Files": round(pct_complete, 2),
        "Unique_Z": num_unique_z,
        "Uniform_dZ": uniform_z,
        "Z_Min": valid_zs[0] if valid_zs else None,
        "Z_Max": valid_zs[-1] if valid_zs else None,
        "Declared_NTP": declared_ntp,
        "Complete_48_Vols": len(complete_48_tpis),
        "Contig_48_Vols": c48_len,
        "Best_Start": c48_start,
        "Best_End": c48_end,
        "GE_10A2_Vals": len(ge_10a2_vals),
        "Category": category,
        "Salvage_Method": salvage_method,
        "Salvageable": salvageable,
        "Target_Dims": target_dims,
        "DCM_Dir": str(dcm_dir),
    }


rows_to_check = [r for _, r in failed_fmri_df.iterrows()]
with ThreadPoolExecutor(max_workers=8) as ex:
    analysis_rows = list(ex.map(analyze_single_series, rows_to_check))

analysis_df = pd.DataFrame(analysis_rows)
ANALYSIS_CSV = (
    PROJECT_ROOT / "results" / "dataset_creation" / "failed_35_fmri_analysis.csv"
)
analysis_df.to_csv(ANALYSIS_CSV, index=False)

print("\n=== Summary by Category ===")
print(analysis_df.groupby(["Salvageable", "Category"]).size().to_string())
print(f"\nTotal Salvageable:   {analysis_df['Salvageable'].sum()} / {len(analysis_df)}")
print(
    f"Total Unsalvageable: {(~analysis_df['Salvageable']).sum()} / {len(analysis_df)}\n"
)

display_cols = [
    "Subject",
    "BIDS_Session",
    "fMRI_Image_ID",
    "Vendor",
    "Total_DCM",
    "Pct_Files",
    "Unique_Z",
    "Complete_48_Vols",
    "Contig_48_Vols",
    "Salvageable",
    "Target_Dims",
    "Category",
]
print(analysis_df[display_cols].to_string(index=False))

Analyzing all 35 failed fMRI series...

=== Summary by Category ===
Salvageable  Category                                                                         
False        Cat 5: Unsalvageable — Internal Z-gap (dz=[np.float64(3.4), np.float64(10.2)] mm)    1
             Cat 5: Unsalvageable — Scattered dropouts (max 2 contig vols)                        1
             Cat 5: Unsalvageable — Severely incomplete (182/9456 = 1.9% files)                   1
             Cat 5: Unsalvageable — Z-truncated (10/48 slices, 20.3% files)                       1
             Cat 5: Unsalvageable — Z-truncated (11/48 slices, 22.8% files)                       1
             Cat 5: Unsalvageable — Z-truncated (15/48 slices, 30.6% files)                       1
             Cat 5: Unsalvageable — Z-truncated (27/48 slices, 56.0% files)                       1
             Cat 5: Unsalvageable — Z-truncated (31/48 slices, 63.2% files)                       1
             Cat 5: Unsalvageable — Z

## 3. Execute Salvage Pipeline on All 24 Recoverable fMRI Scans & Clean Up Orphaned Files

For each of the 35 failed fMRI series:
- First, we remove any orphaned 1-volume `.nii.gz` files in `data/raw_bids/sub-.../ses-.../func/` left behind when `dcm2niix` aborted with `rc=1` on the 9 GE scans.
- For the **24 salvageable scans**, we stage the repaired/subsetted DICOM series in a temporary folder on `D:\` (`data/_tmp_salvage`), run `dcm2niix`, inject `"TaskName": "rest"` into the BIDS `.json` sidecar, and verify the 4D NIfTI dimensions `(X, Y, Z, T)`.
- Finally, we update `bids_conversion_manifest.csv` and save `results/dataset_creation/fmri_salvage_results.csv`.

In [ ]:
TMP_SALVAGE_DIR = PROJECT_ROOT / "data" / "_tmp_salvage"


def salvage_fmri_series(row: pd.Series) -> dict:
    subj = row["Subject"]
    ses = row["BIDS_Session"]
    img_id = row["fMRI_Image_ID"]
    method = row["Salvage_Method"]
    dcm_dir = Path(row["DCM_Dir"])

    bids_sub = "sub-" + subj.replace("_", "")
    func_dir = RAW_BIDS_DIR / bids_sub / ses / "func"
    func_prefix = f"{bids_sub}_{ses}_task-rest_bold"

    # Always clean up any partial/orphaned outputs from previous failed run
    if func_dir.exists():
        for old_f in func_dir.glob(f"{func_prefix}*"):
            old_f.unlink()

    if not row["Salvageable"]:
        # Remove empty func_dir if nothing is inside
        if func_dir.exists() and not any(func_dir.iterdir()):
            func_dir.rmdir()
        return {
            "Subject": subj,
            "BIDS_Session": ses,
            "fMRI_Image_ID": img_id,
            "Salvaged": False,
            "Method": method,
            "Output_Dims": None,
            "Retained_Vols": 0,
            "Synthesized_Slices": 0,
            "Status": f"Unsalvageable ({row['Category']})",
        }

    func_dir.mkdir(parents=True, exist_ok=True)
    if TMP_SALVAGE_DIR.exists():
        shutil.rmtree(TMP_SALVAGE_DIR)
    tmp_in = TMP_SALVAGE_DIR / "in"
    tmp_in.mkdir(parents=True, exist_ok=True)

    dcm_files = sorted(dcm_dir.glob("*.dcm"))
    synth_count = 0
    staged_count = 0

    try:
        if method == "strip_ge_10a2":
            for i, f in enumerate(dcm_files):
                ds = pydicom.dcmread(f)
                if (0x0019, 0x10A2) in ds:
                    del ds[(0x0019, 0x10A2)]
                ds.save_as(tmp_in / f"img_{i:05d}.dcm")
                staged_count += 1

        elif method == "interp_single_slices":
            is_ge = row["Vendor"] == "GE"
            ntp = int(row["Declared_NTP"])
            grid = {}
            tpi_meta = {}
            z_set = set()

            for i, f in enumerate(dcm_files):
                ds_hdr = pydicom.dcmread(f, stop_before_pixels=True)
                tpi = getattr(ds_hdr, "TemporalPositionIdentifier", None)
                ipp = getattr(ds_hdr, "ImagePositionPatient", None)
                if tpi is None or ipp is None:
                    continue
                tpi = int(tpi)
                z = round(float(ipp[2]), 2)
                out_f = tmp_in / f"orig_{staged_count:05d}.dcm"
                if is_ge:
                    ds_full = pydicom.dcmread(f)
                    if (0x0019, 0x10A2) in ds_full:
                        del ds_full[(0x0019, 0x10A2)]
                    ds_full.save_as(out_f)
                else:
                    os.link(f, out_f)
                staged_count += 1
                grid[(z, tpi)] = out_f
                z_set.add(z)
                if tpi not in tpi_meta:
                    tpi_meta[tpi] = ds_hdr

            all_zs = sorted(z_set)
            for z_idx, z in enumerate(all_zs):
                for t in range(1, ntp + 1):
                    if (z, t) not in grid:
                        t_prev = max(
                            [tp for tp in range(1, t) if (z, tp) in grid], default=None
                        )
                        t_next = min(
                            [tp for tp in range(t + 1, ntp + 1) if (z, tp) in grid],
                            default=None,
                        )
                        donor_t = t_prev if t_prev is not None else t_next
                        ds_new = pydicom.dcmread(grid[(z, donor_t)])
                        if t_prev is not None and t_next is not None:
                            ds_prev = pydicom.dcmread(grid[(z, t_prev)])
                            ds_next = pydicom.dcmread(grid[(z, t_next)])
                            arr = (
                                (
                                    ds_prev.pixel_array.astype(np.int32)
                                    + ds_next.pixel_array.astype(np.int32)
                                )
                                // 2
                            ).astype(ds_prev.pixel_array.dtype)
                            ds_new.PixelData = arr.tobytes()
                        ds_new.TemporalPositionIdentifier = t
                        ds_new.SOPInstanceUID = generate_uid()
                        if hasattr(ds_new, "file_meta") and hasattr(
                            ds_new.file_meta, "MediaStorageSOPInstanceUID"
                        ):
                            ds_new.file_meta.MediaStorageSOPInstanceUID = (
                                ds_new.SOPInstanceUID
                            )
                        if is_ge:
                            in_stack = int(
                                getattr(ds_new, "InStackPositionNumber", z_idx + 1)
                            )
                            ds_new.InstanceNumber = (t - 1) * len(all_zs) + in_stack
                        else:
                            ds_new.InstanceNumber = int(ds_new.InstanceNumber) + (
                                t - donor_t
                            )
                        ref_ds = tpi_meta[t]
                        for tag_name in [
                            "AcquisitionTime",
                            "ContentTime",
                            "TriggerTime",
                            "AcquisitionDateTime",
                        ]:
                            if hasattr(ref_ds, tag_name):
                                setattr(ds_new, tag_name, getattr(ref_ds, tag_name))
                        ds_new.save_as(
                            tmp_in / f"synth_{synth_count:03d}_z{z_idx}_t{t}.dcm"
                        )
                        synth_count += 1
                        staged_count += 1

        elif method == "contiguous_48_subset":
            is_ge = row["Vendor"] == "GE"
            t_start = int(row["Best_Start"])
            t_end = int(row["Best_End"])
            for f in dcm_files:
                ds_hdr = pydicom.dcmread(f, stop_before_pixels=True)
                tpi = getattr(ds_hdr, "TemporalPositionIdentifier", None)
                if tpi is not None and t_start <= int(tpi) <= t_end:
                    out_f = tmp_in / f"img_{staged_count:05d}.dcm"
                    if is_ge:
                        ds_full = pydicom.dcmread(f)
                        if (0x0019, 0x10A2) in ds_full:
                            del ds_full[(0x0019, 0x10A2)]
                        ds_full.save_as(out_f)
                    else:
                        os.link(f, out_f)
                    staged_count += 1

        elif method == "slab_tpi_subset":
            # 019_S_6315 I1314046: TPI 1..132 has all 47 slices
            for f in dcm_files:
                ds_hdr = pydicom.dcmread(f, stop_before_pixels=True)
                tpi = getattr(ds_hdr, "TemporalPositionIdentifier", None)
                if tpi is not None and 1 <= int(tpi) <= 132:
                    os.link(f, tmp_in / f"img_{staged_count:05d}.dcm")
                    staged_count += 1

        elif method == "slab_z_subset":
            # 002_S_0413 I569636: keep the 44 slices that have all 200 TPIs
            ntp = int(row["Declared_NTP"])
            z_files = defaultdict(list)
            for f in dcm_files:
                ds_hdr = pydicom.dcmread(f, stop_before_pixels=True)
                z = round(float(ds_hdr.ImagePositionPatient[2]), 2)
                z_files[z].append(f)
            for z, flist in sorted(z_files.items()):
                if len(flist) == ntp:
                    for f in flist:
                        os.link(f, tmp_in / f"img_{staged_count:05d}.dcm")
                        staged_count += 1

        # Run dcm2niix
        res = subprocess.run(
            [
                DCM2NIIX_BIN,
                "-z",
                "y",
                "-b",
                "y",
                "-f",
                func_prefix,
                "-o",
                str(func_dir),
                str(tmp_in),
            ],
            capture_output=True,
            text=True,
        )
        nii_path = func_dir / f"{func_prefix}.nii.gz"
        json_path = func_dir / f"{func_prefix}.json"

        if res.returncode == 0 and nii_path.exists() and json_path.exists():
            # Inject TaskName: rest into BIDS JSON sidecar
            meta = json.loads(json_path.read_text(encoding="utf-8"))
            if "TaskName" not in meta:
                meta["TaskName"] = "rest"
                json_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
            dims = read_nifti_dims(nii_path)
            status = (
                f"OK (Salvaged via {method}: dims={dims}, synth_slices={synth_count})"
            )
            return {
                "Subject": subj,
                "BIDS_Session": ses,
                "fMRI_Image_ID": img_id,
                "Salvaged": True,
                "Method": method,
                "Output_Dims": str(dims),
                "Retained_Vols": int(dims[3]) if len(dims) >= 4 else 1,
                "Synthesized_Slices": synth_count,
                "Status": status,
            }
        else:
            return {
                "Subject": subj,
                "BIDS_Session": ses,
                "fMRI_Image_ID": img_id,
                "Salvaged": False,
                "Method": method,
                "Output_Dims": None,
                "Retained_Vols": 0,
                "Synthesized_Slices": synth_count,
                "Status": f"ERROR rc={res.returncode}",
            }
    finally:
        if TMP_SALVAGE_DIR.exists():
            shutil.rmtree(TMP_SALVAGE_DIR)


t0 = time.time()
salvage_results = []
for idx, row in analysis_df.iterrows():
    r = salvage_fmri_series(row)
    salvage_results.append(r)
    tag = "SALVAGED" if r["Salvaged"] else "SKIPPED "
    print(
        f"[{len(salvage_results):02d}/35] {tag} | {r['Subject']} {r['BIDS_Session']} {r['fMRI_Image_ID']:8s} | {r['Status']}"
    )

salvage_df = pd.DataFrame(salvage_results)
SALVAGE_CSV = PROJECT_ROOT / "results" / "dataset_creation" / "fmri_salvage_results.csv"
salvage_df.to_csv(SALVAGE_CSV, index=False)
print(
    f"\nFinished salvage pipeline in {time.time() - t0:.1f}s. Saved results to {SALVAGE_CSV.relative_to(PROJECT_ROOT)}"
)

[01/35] SALVAGED | 002_S_0413 ses-20150609 I569636  | OK (Salvaged via slab_z_subset: dims=(64, 64, 44, 200), synth_slices=0)
[02/35] SKIPPED  | 002_S_1261 ses-20140313 I418009  | Unsalvageable (Cat 5: Unsalvageable — Z-truncated (6/48 slices, 10.9% files))
[03/35] SKIPPED  | 006_S_0731 ses-20181010 I1058017 | Unsalvageable (Cat 5: Unsalvageable — Z-truncated (27/48 slices, 56.0% files))
[04/35] SKIPPED  | 006_S_6252 ses-20190506 I1162267 | Unsalvageable (Cat 5: Unsalvageable — Severely incomplete (182/9456 = 1.9% files))
[05/35] SALVAGED | 006_S_6277 ses-20190425 I1158788 | OK (Salvaged via interp_single_slices: dims=(64, 64, 48, 197), synth_slices=1)
[06/35] SKIPPED  | 006_S_6500 ses-20180730 I1028502 | Unsalvageable (Cat 5: Unsalvageable — Internal Z-gap (dz=[np.float64(3.4), np.float64(10.2)] mm))
[07/35] SALVAGED | 006_S_6610 ses-20191126 I1260257 | OK (Salvaged via interp_single_slices: dims=(64, 64, 48, 197), synth_slices=1)
[08/35] SKIPPED  | 006_S_6681 ses-20190212 I1148835 | 

## 4. Update `bids_conversion_manifest.csv` & Verify Final BIDS Dataset Integrity

We update `bids_conversion_manifest.csv` with the 24 newly salvaged fMRI series (and the fixed T1w `I232233` scan), verify on-disk file completeness across all converted sessions, and print the final summary report.

In [ ]:
# Update manifest_df with salvage results
for _, srow in salvage_df.iterrows():
    mask = (
        (manifest_df["Subject"] == srow["Subject"])
        & (manifest_df["BIDS_Session"] == srow["BIDS_Session"])
        & (manifest_df["fMRI_Image_ID"] == srow["fMRI_Image_ID"])
    )
    if srow["Salvaged"]:
        manifest_df.loc[mask, "fMRI_Converted"] = True
        manifest_df.loc[mask, "fMRI_Status"] = srow["Status"]
    else:
        manifest_df.loc[mask, "fMRI_Converted"] = False
        manifest_df.loc[mask, "fMRI_Status"] = srow["Status"]

manifest_df.to_csv(MANIFEST_CSV, index=False)
both_ok_final = manifest_df["fMRI_Converted"] & manifest_df["T1_Converted"]

# Verify no broken 1-volume fMRI NIfTIs remain anywhere in data/raw_bids
orphaned_niis = []
for func_nii in RAW_BIDS_DIR.rglob("*_bold.nii.gz"):
    json_sidecar = func_nii.with_name(func_nii.name.replace(".nii.gz", ".json"))
    dims = read_nifti_dims(func_nii)
    if not json_sidecar.exists() or len(dims) < 4 or dims[3] == 1:
        orphaned_niis.append((str(func_nii), dims, json_sidecar.exists()))

print("=== Final BIDS Conversion Summary (3,278 Total Sessions) ===")
print(
    f"  T1w Converted:              {manifest_df['T1_Converted'].sum():,} / {len(manifest_df):,} ({manifest_df['T1_Converted'].mean() * 100:.2f}%)"
)
print(
    f"  fMRI Converted:             {manifest_df['fMRI_Converted'].sum():,} / {len(manifest_df):,} ({manifest_df['fMRI_Converted'].mean() * 100:.2f}%)"
)
print(
    f"  Complete Pairs (Both OK):   {both_ok_final.sum():,} / {len(manifest_df):,} ({both_ok_final.mean() * 100:.2f}%)"
)
print(
    f"  Unrecoverable fMRI Scans:   {(~manifest_df['fMRI_Converted']).sum()} / {len(manifest_df):,} ({(~manifest_df['fMRI_Converted']).mean() * 100:.2f}%)"
)
print(f"  Broken/Orphaned 1-Vol BOLD NIfTIs remaining on disk: {len(orphaned_niis)}")

print("\n=== 24 Salvaged fMRI Scans ===")
print(
    salvage_df[salvage_df["Salvaged"]][
        [
            "Subject",
            "BIDS_Session",
            "fMRI_Image_ID",
            "Method",
            "Output_Dims",
            "Retained_Vols",
            "Synthesized_Slices",
        ]
    ].to_string(index=False)
)

print("\n=== 11 Truly Unsalvageable fMRI Scans (Excluded) ===")
unsalv_merged = analysis_df[~analysis_df["Salvageable"]][
    [
        "Subject",
        "BIDS_Session",
        "fMRI_Image_ID",
        "Vendor",
        "Total_DCM",
        "Expected_DCM",
        "Pct_Files",
        "Unique_Z",
        "Category",
    ]
]
print(unsalv_merged.to_string(index=False))

=== Final BIDS Conversion Summary (3,278 Total Sessions) ===
  T1w Converted:              3,278 / 3,278 (100.00%)
  fMRI Converted:             3,267 / 3,278 (99.66%)
  Complete Pairs (Both OK):   3,267 / 3,278 (99.66%)
  Unrecoverable fMRI Scans:   11 / 3,278 (0.34%)
  Broken/Orphaned 1-Vol BOLD NIfTIs remaining on disk: 0

=== 24 Salvaged fMRI Scans ===
   Subject BIDS_Session fMRI_Image_ID               Method       Output_Dims  Retained_Vols  Synthesized_Slices
002_S_0413 ses-20150609       I569636        slab_z_subset (64, 64, 44, 200)            200                   0
006_S_6277 ses-20190425      I1158788 interp_single_slices (64, 64, 48, 197)            197                   1
006_S_6610 ses-20191126      I1260257 interp_single_slices (64, 64, 48, 197)            197                   1
006_S_6689 ses-20220331      I1563081 contiguous_48_subset (64, 64, 48, 166)            166                   0
006_S_6727 ses-20190507      I1162444 interp_single_slices (64, 64, 48, 197)     

## 5. Add Descriptive `Comments` Column to `bids_conversion_manifest.csv`

To support downstream quality control and ablation studies (e.g., training/evaluating with vs. without temporally interpolated scans, volume-subsetted scans, or Z-slab-cropped scans), we add a `Comments` column to `bids_conversion_manifest.csv` documenting the exact conversion history of every session.

In [ ]:
# Build detailed Comments column for bids_conversion_manifest.csv
manifest_df = pd.read_csv(MANIFEST_CSV)

CUSTOM_COMMENTS = {
    # 1. T1w misassignment fix (fMRI converted first try)
    "I232243": "fMRI converted with no problems first try; T1w replaced misassigned localizer I232236 with true MPRAGE I232233",
    # 2. Cat 1 — GE Private Tag (0019,10A2) Bug (5 scans, 100% intact, 0 interpolated slices)
    "I680900": "salvaged (tag fix only): stripped erroneous per-slice GE tag (0019,10A2); all 200 volumes x 48 slices intact (0 interpolated slices)",
    "I638776": "salvaged (tag fix only): stripped erroneous per-slice GE tag (0019,10A2); all 200 volumes x 48 slices intact (0 interpolated slices)",
    "I631785": "salvaged (tag fix only): stripped erroneous per-slice GE tag (0019,10A2); all 200 volumes x 48 slices intact (0 interpolated slices)",
    "I695057": "salvaged (tag fix only): stripped erroneous per-slice GE tag (0019,10A2); all 200 volumes x 48 slices intact (0 interpolated slices)",
    "I680924": "salvaged (tag fix only): stripped erroneous per-slice GE tag (0019,10A2); all 200 volumes x 48 slices intact (0 interpolated slices)",
    # 3. Cat 2 — Temporally interpolated single-slice PACS dropouts (13 scans)
    "I1158788": "salvaged (interpolated 1 slice): slice 40 interpolated in volume 97 (also dropped 1 extra localizer file with TPI=None); 197 volumes x 48 slices",
    "I1260257": "salvaged (interpolated 1 slice): slice 9 interpolated in volume 12; 197 volumes x 48 slices",
    "I1162444": "salvaged (interpolated 1 slice): slice 1 interpolated in volume 184; 197 volumes x 48 slices",
    "I1011356": "salvaged (interpolated 1 slice): slice 22 interpolated in volume 47; 197 volumes x 48 slices",
    "I1416249": "salvaged (interpolated 1 slice + GE tag fix): slice 42 interpolated in volume 73, stripped GE tag (0019,10A2); 200 volumes x 48 slices",
    "I1345126": "salvaged (interpolated 1 slice): slice 26 interpolated in volume 65; 197 volumes x 48 slices",
    "I1441195": "salvaged (interpolated 1 slice): slice 25 interpolated in volume 90; 197 volumes x 48 slices",
    "I1129771": "salvaged (interpolated 1 slice): slice 23 interpolated in volume 176; 197 volumes x 48 slices",
    "I1304676": "salvaged (interpolated 3 slices): (slice 1, vol 16), (slice 29, vol 119), (slice 29, vol 120) interpolated; 197 volumes x 48 slices",
    "I1367893": "salvaged (interpolated 5 slices + GE tag fix): (slice 21, vol 117), (slice 30, vol 118), (slice 40, vol 118), (slice 36, vol 194), (slice 9, vol 195) interpolated, stripped GE tag (0019,10A2); 200 volumes x 48 slices",
    "I1162410": "salvaged (interpolated 7 slices): slice 46 interpolated in volumes [18, 71, 73, 82, 85] and slice 8 interpolated in volumes [123, 127]; 197 volumes x 48 slices",
    "I999077": "salvaged (interpolated 8 slices): (slice 38, vols 11,14), (slice 20, vols 71,74), (slice 30, vol 91), (slice 13, vols 123,128), (slice 23, vol 139) interpolated; 197 volumes x 48 slices",
    "I1153522": "salvaged (interpolated 9 slices): (slice 17, vol 17), (slice 40, vol 66), (slice 41, vols 78,116), (slice 48, vol 81), (slice 13, vol 89), (slice 34, vol 101), (slice 46, vol 162), (slice 7, vol 200) interpolated; 200 volumes x 48 slices",
    # 4. Cat 3 — Contiguous complete-volume subsetting (4 scans, 0 interpolated slices)
    "I880134": "salvaged (dropped incomplete volume + GE tag fix): kept volumes 1..167 (167 vols x 48 slices); dropped last volume 168 (incomplete, 23 slices missing; vols 169..200 not acquired)",
    "I1386024": "salvaged (dropped incomplete tail volumes + GE tag fix): kept volumes 1..137 (137 vols x 48 slices); dropped tail volumes 138..142 (incomplete, 1 to 17 slices missing; vols 143..200 not acquired)",
    "I1563081": "salvaged (dropped incomplete early volumes): kept contiguous volumes 32..197 (166 vols x 48 slices); dropped volumes 1..31 due to multi-slice dropouts in vols 19, 22, 27, 31 (8 to 18 slices missing)",
    "I1037522": "salvaged (dropped incomplete boundary volumes): kept contiguous volumes 27..124 (98 vols x 48 slices); dropped vols 1..26 (slice 25 missing in 9 vols) and vols 125..197 (slice 24 missing in 25 vols)",
    # 5. Cat 4 — Near-complete axial Z-slab subsetting (2 scans, 0 interpolated slices)
    "I1314046": "salvaged (cropped Z-slab + volume subset): kept volumes 1..132 (132 vols x 47/48 slices, missing top vertex slice 48); dropped volumes 133..197 (missing slice 47 in vols 133..197)",
    "I569636": "salvaged (cropped Z-slab): kept slices 1..44 across all 200 volumes (200 vols x 44/48 slices); dropped top vertex slice 45 (incomplete, 143 timepoints missing) and slices 46..48 (missing)",
    # 6. Cat 5 — Unsalvageable excluded scans (11 scans)
    "I1162267": "failed (unsalvageable): severely incomplete upload, only 182/9456 slice files (1.9%) present across 48 slices (max 4 contiguous volumes)",
    "I418009": "failed (unsalvageable): slice-major upload truncated at 1046/9600 files (10.9%); only 5/48 complete axial slices present (42 slices completely missing)",
    "I431143": "failed (unsalvageable): slice-major upload truncated at 1945/9600 files (20.3%); only 9/48 complete axial slices present (38 slices completely missing)",
    "I272899": "failed (unsalvageable): slice-major upload truncated at 1533/6720 files (22.8%); only 10/48 complete axial slices present (37 slices completely missing)",
    "I1055464": "failed (unsalvageable): slice-major upload truncated at 2938/9600 files (30.6%); only 14/48 complete axial slices present (33 slices completely missing)",
    "I1058017": "failed (unsalvageable): slice-major upload truncated at 5291/9456 files (56.0%); only 26/48 complete axial slices present (21 slices completely missing)",
    "I1148835": "failed (unsalvageable): slice-major upload truncated at 5974/9456 files (63.2%); only 30/48 complete axial slices present (17 slices completely missing)",
    "I1504396": "failed (unsalvageable): slice-major upload truncated at 6448/9456 files (68.2%); only 32/48 complete axial slices present (15 slices completely missing)",
    "I659915": "failed (unsalvageable): severe random slice dropouts across 170/200 volumes (2499/9600 slices missing, max 2 contiguous complete volumes)",
    "I335381": "failed (unsalvageable): slice-major upload truncated at 5737/6720 files (85.4%); only 40/48 complete axial slices present (top 7 cortical slices missing)",
    "I1028502": "failed (unsalvageable): missing 2 internal cortical slices between Z=70.4mm and Z=80.6mm (10.2mm spatial gap) plus incomplete slices 31..32",
}

manifest_df["Comments"] = (
    manifest_df["fMRI_Image_ID"]
    .map(CUSTOM_COMMENTS)
    .fillna("converted with no problems first try")
)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print(
    f"Updated {MANIFEST_CSV.relative_to(PROJECT_ROOT)} with 'Comments' column ({len(manifest_df):,} total rows)."
)
print("\n=== Breakdown by Conversion Comment Type ===")
comment_type = manifest_df["Comments"].apply(
    lambda s: s.split(":")[0] if ":" in s else s
)
print(comment_type.value_counts().to_string())

print("\n=== All 36 Non-Default Sessions in bids_conversion_manifest.csv ===")
non_default = manifest_df[
    manifest_df["Comments"] != "converted with no problems first try"
][
    [
        "Subject",
        "BIDS_Session",
        "fMRI_Image_ID",
        "T1_Image_ID",
        "fMRI_Converted",
        "T1_Converted",
        "Comments",
    ]
]
print(non_default.to_string(index=False))

Updated results\dataset_creation\bids_conversion_manifest.csv with 'Comments' column (3,278 total rows).

=== Breakdown by Conversion Comment Type ===
Comments
converted with no problems first try                                                                              3242
failed (unsalvageable)                                                                                              11
salvaged (interpolated 1 slice)                                                                                      7
salvaged (tag fix only)                                                                                              5
salvaged (cropped Z-slab)                                                                                            1
salvaged (dropped incomplete early volumes)                                                                          1
salvaged (dropped incomplete volume + GE tag fix)                                                                    1
salvage